# Замороженный MPNet и CatBoost

Проверяем, помогают ли готовые смысловые представления отзывов предсказывать оценку 1–5. Энкодер sentence-transformers/all-mpnet-base-v2 не дообучается: он один раз превращает текст в 768 чисел. CatBoostRegressor обучается на этих векторах с функцией потерь MAE. Разбиение совпадает с baselinev2.ipynb: валидационные тексты не пересекаются с обучающими.

Числа и таблицы ниже сохранены из предыдущего запуска на настоящих данных. После разбиения длинных ячеек обучение повторно не запускалось.

In [ ]:
import gc
import json
import os
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostRegressor
from IPython.display import display
from sentence_transformers import SentenceTransformer

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedGroupKFold


def cache_digest(config: dict) -> str:
    """Короткий устойчивый ключ для JSON-совместимых параметров эксперимента."""
    import json
    from hashlib import sha256

    payload = json.dumps(
        config, sort_keys=True, separators=(",", ":"), ensure_ascii=False
    )
    return sha256(payload.encode("utf-8")).hexdigest()[:12]


root = Path.cwd()

if not (root / "data" / "raw" / "train.csv").exists():
    root = root.parent

data_path = root / "data" / "raw" / "train.csv"

if not data_path.exists():
    raise FileNotFoundError("Поместите train.csv в data/raw/")

train = pd.read_csv(data_path)

In [ ]:
assert list(train.columns) == ["Id", "Review", "Rating"]

clean = train.drop_duplicates(subset=["Review"], keep="first").reset_index(drop=True)

groups = clean["Review"].str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2026)

train_idx, valid_idx = next(splitter.split(clean["Review"], clean["Rating"], groups))

assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[valid_idx]))

y_train = clean["Rating"].iloc[train_idx].to_numpy()

y_valid = clean["Rating"].iloc[valid_idx].to_numpy()

print(f"Обучение: {len(train_idx):,}; валидация: {len(valid_idx):,}")

Обучение: 47,980; валидация: 11,996

## Векторизация отзывов

Используем закреплённую версию MPNet. На компьютере с 8 ГБ памяти ограничиваем вход 256 токенами и кодируем небольшими пакетами. Кэш в `data/cache/` зависит от фактического набора очищенных текстов, версии энкодера, длины входа и параметров кодирования. При изменении этих условий эмбеддинги вычисляются заново.

In [ ]:
model_id = "sentence-transformers/all-mpnet-base-v2"

model_revision = "e8c3b32edf5434bc2275fc9bab85f82640a19130"

max_length = 256

encode_batch_size = 16

normalize_embeddings = True

device = "mps" if torch.backends.mps.is_available() else "cpu"

data_sha256 = sha256(data_path.read_bytes()).hexdigest()

review_hash = sha256()

for review in clean["Review"]:
    review_hash.update(review.encode("utf-8"))
    review_hash.update(b"\0")

split_sha256 = sha256(
    train_idx.astype("<i8").tobytes() + valid_idx.astype("<i8").tobytes()
).hexdigest()

In [ ]:
embedding_config = {
    "reviews_sha256": review_hash.hexdigest(),
    "model_id": model_id,
    "model_revision": model_revision,
    "max_length": max_length,
    "encode_batch_size": encode_batch_size,
    "normalize_embeddings": normalize_embeddings,
    "device": device,
    "sentence_transformers_version": version("sentence-transformers"),
}

embedding_digest = cache_digest(embedding_config)

cache_dir = root / "data" / "cache"

cache_dir.mkdir(parents=True, exist_ok=True)

cache_path = cache_dir / f"mpnet_{embedding_digest}.npy"

embedding_manifest = cache_dir / f"mpnet_{embedding_digest}.json"

expected_shape = (len(clean), 768)

In [ ]:
if cache_path.exists() != embedding_manifest.exists():
    raise RuntimeError(
        "Кэш MPNet неполный: удалите соответствующие .npy и .json из data/cache/"
    )

needs_encoding = not (cache_path.exists())

if not needs_encoding:
    if json.loads(embedding_manifest.read_text(encoding="utf-8")) != embedding_config:
        raise RuntimeError("Параметры кэша MPNet не совпадают с текущим запуском")
    print("Использован локальный кэш эмбеддингов")

In [ ]:
if needs_encoding:
    started = perf_counter()

    encoder = SentenceTransformer(model_id, revision=model_revision, device=device)

    encoder.max_seq_length = max_length

    partial_path = cache_path.with_suffix(".partial.npy")

    vectors = np.lib.format.open_memmap(
        partial_path, mode="w+", dtype=np.float32, shape=expected_shape
    )

    chunk_size = 256

    for start in range(0, len(clean), chunk_size):
        stop = min(start + chunk_size, len(clean))
        vectors[start:stop] = encoder.encode(
            clean["Review"].iloc[start:stop].tolist(),
            batch_size=encode_batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=normalize_embeddings,
        )
        if device == "mps":
            torch.mps.empty_cache()
        if stop % 5120 < chunk_size or stop == len(clean):
            vectors.flush()
            print(f"Закодировано {stop:,} / {len(clean):,}", flush=True)

    vectors.flush()

In [ ]:
if needs_encoding:
    del vectors, encoder

    gc.collect()

    if device == "mps":
        torch.mps.empty_cache()

    os.replace(partial_path, cache_path)

    embedding_manifest.write_text(
        json.dumps(embedding_config, sort_keys=True, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"Время кодирования: {(perf_counter() - started) / 60:.1f} мин")

embeddings = np.load(cache_path, mmap_mode="r", allow_pickle=False)

assert embeddings.shape == expected_shape

assert embeddings.dtype == np.float32

print("Размер матрицы:", embeddings.shape, "устройство:", device)

Время кодирования: 17.8 мин

Размер матрицы: (59976, 768) устройство: mps

## Обучение регрессора

CatBoost получает только эмбеддинги обучающих отзывов и их оценки. Путь к кэшу зависит от эмбеддингов, точного разбиения и параметров CatBoost: изменение настройки требует нового обучения. Выход ограничиваем диапазоном 1–5; округление не требуется, поскольку метрика Kaggle принимает дробные предсказания.

In [ ]:
X_train = np.asarray(embeddings[train_idx], dtype=np.float32)

X_valid = np.asarray(embeddings[valid_idx], dtype=np.float32)

assert X_train.shape == (len(train_idx), 768)

assert X_valid.shape == (len(valid_idx), 768)

regressor_params = {
    "loss_function": "MAE",
    "iterations": 500,
    "depth": 6,
    "learning_rate": 0.05,
    "thread_count": 4,
    "random_seed": 2026,
    "verbose": False,
    "allow_writing_files": False,
}

regressor_config = {
    "data_sha256": data_sha256,
    "embedding_digest": embedding_digest,
    "split_sha256": split_sha256,
    "catboost_version": version("catboost"),
    "parameters": regressor_params,
}

In [ ]:
regressor_digest = cache_digest(regressor_config)

regressor_path = cache_dir / f"catboost_mpnet_{regressor_digest}.cbm"

regressor_manifest = cache_dir / f"catboost_mpnet_{regressor_digest}.json"

regressor = CatBoostRegressor(**regressor_params)

if regressor_path.exists() != regressor_manifest.exists():
    raise RuntimeError(
        "Кэш CatBoost неполный: удалите соответствующие .cbm и .json из data/cache/"
    )

needs_regression = not (regressor_path.exists())

if not needs_regression:
    if json.loads(regressor_manifest.read_text(encoding="utf-8")) != regressor_config:
        raise RuntimeError("Параметры кэша CatBoost не совпадают с текущим запуском")
    regressor.load_model(str(regressor_path))
    print("Использован локальный кэш CatBoost")

In [ ]:
if needs_regression:
    started = perf_counter()

    regressor.fit(X_train, y_train)

    partial_path = regressor_path.with_suffix(".partial.cbm")

    regressor.save_model(str(partial_path))

    os.replace(partial_path, regressor_path)

    regressor_manifest.write_text(
        json.dumps(regressor_config, sort_keys=True, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"Обучение CatBoost: {(perf_counter() - started) / 60:.1f} мин")

predictions = np.clip(regressor.predict(X_valid), 1, 5)

mae = mean_absolute_error(y_valid, predictions)

print(f"MAE MPNet + CatBoost: {mae:.5f}")

In [ ]:
by_rating = pd.DataFrame(
    {
        "строк": [int((y_valid == rating).sum()) for rating in range(1, 6)],
        "MAE": [
            mean_absolute_error(
                y_valid[y_valid == rating], predictions[y_valid == rating]
            )
            for rating in range(1, 6)
        ],
    },
    index=pd.Index(range(1, 6), name="Rating"),
)

display(by_rating.round(4))

Обучение CatBoost: 0.5 мин

MAE MPNet + CatBoost: 0.44325

| Rating | строк | MAE |
| --- | --- | --- |
| 1 | 3732 | 0.5765 |
| 2 | 326 | 1.0141 |
| 3 | 336 | 1.1681 |
| 4 | 670 | 0.8963 |
| 5 | 6932 | 0.2657 |

## Выводы

- MPNet + CatBoost дал **MAE 0,44325** на отложенной части. TF-IDF + Logistic Regression с `C=4`, `min_df=5` из `baselinev2.ipynb` получил **0,18881 на тех же 11 996 отзывах**; разница составляет 0,25444 MAE. Среднюю MAE пяти фолдов из `baselinev2.ipynb` здесь не используем, поскольку это другая схема оценки.
- По оценкам 1 и 5 MAE составила 0,5765 и 0,2657 соответственно. По редким оценкам 2–4 модель также часто ошибается; общий результат нельзя объяснить только их малой численностью.
- Получение векторов заняло 17,8 мин на Apple M1; обучение CatBoost — 0,5 мин. Это измерения этого запуска на данном компьютере, а не универсальные оценки скорости.
- Энкодер не адаптировался к задаче рейтинга. Возможное объяснение слабого результата — общая смысловая близость отзывов не всегда отражает разницу между оценками. В `deberta.ipynb` рассматриваем отдельный вариант — частичное дообучение трансформера. Поскольку меняются энкодер и способ обучения, этот опыт не позволяет установить причину результата MPNet.